# Defining Tools with MCP

Building an MCP server gets much simpler with the official Python SDK. Instead of hand-writing JSON schemas, you define tools with **decorators** and let the SDK generate the schema from your type hints and field descriptions.

We're building a **document management server** with two tools: one to **read** a document, one to **edit** it. Documents live in memory as a dict keyed by document id.

> Implement these in **`cli-project/mcp_server.py`** (replace the first two `TODO`s). That file is `skip-worktree`, so your edits stay local. The finished version is in `cli-project-complete/`.

## Setting up the server

The SDK makes server creation a one-liner:

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DocumentMCP", log_level="ERROR")
```

Documents in a simple dict:

```python
docs = {
    "deposition.md": "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "financials.docx": "These financials outline the project's budget and expenditures",
    "outlook.pdf": "This document presents the projected future performance of the system",
    "plan.md": "The plan outlines the steps for the project's implementation.",
    "spec.txt": "These specifications define the technical requirements for the equipment"
}
```

## Tool definition with decorators

The SDK uses `@mcp.tool(...)` to register a tool. Python **type hints** define the arguments; Pydantic's **`Field`** supplies per-argument descriptions. The SDK turns that into the JSON schema Claude sees — no manual schema writing.

```python
from pydantic import Field   # needed for Field(...) descriptions
```

## Document reader tool

Reads a document's contents by id:

```python
@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string."
)
def read_document(
    doc_id: str = Field(description="Id of the document to read")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")

    return docs[doc_id]
```

The decorator sets the tool **name** and **description**; the function parameters define the required arguments. `Field(description=...)` tells Claude what each parameter expects.

## Document editor tool

A simple find-and-replace on a document:

```python
@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the documents content with a new string."
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="The text to replace. Must match exactly, including whitespace."),
    new_str: str = Field(description="The new text to insert in place of the old text.")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")

    docs[doc_id] = docs[doc_id].replace(old_str, new_str)
```

Three parameters — doc id, text to find, replacement text — plus a missing-document guard and a straight string replace.

## Why the SDK approach wins

- No manual JSON schema writing.
- Type hints give automatic validation.
- Clear `Field` descriptions help Claude understand each parameter.
- Python exceptions integrate naturally as tool errors.
- Registration is automatic via the decorator.

The SDK turns tool creation from a schema-writing exercise into ordinary Python functions, while still handing Claude a properly-formatted tool spec.

> Same contract as course 1's tool-use section: Claude only ever sees the **schema**; here the SDK generates it for you instead of you writing the dict by hand.

Next: **The server inspector** — run and exercise the server without wiring up a full client.